# Climate dashboard: data download

This notebook downloads the raw data files for our climate dashboard, then shows a quick preview so you can check them before sending them over.

| File | What it is | Source |
|---|---|---|
| `co2_mm_gl.csv` | Global monthly mean CO₂ (ppm), since 1979 | NOAA Global Monitoring Laboratory |
| `ch4_mm_gl.csv` | Global monthly mean methane, CH₄ (ppb), since 1983 | NOAA Global Monitoring Laboratory |
| `n2o_mm_gl.csv` | Global monthly mean nitrous oxide, N₂O (ppb), since 2001 | NOAA Global Monitoring Laboratory |
| `GLB.Ts+dSST.csv` | Global surface temperature anomaly (°C, vs 1951–1980), since 1880 | NASA GISS, GISTEMP v4 |

**How to run it**

- **Google Colab** (free, nothing to install): go to [colab.research.google.com](https://colab.research.google.com), choose **File → Upload notebook**, pick this file, then **Runtime → Run all**. The last cell downloads everything as one zip.
- **Jupyter or VS Code** on your computer: open the notebook and run all cells. The files land in a `climate_data` folder next to the notebook.

The files are saved exactly as published, with no cleanup. Re-run any time to refresh: NASA updates around the 10th of each month, NOAA roughly monthly.

*Credits:* NOAA Global Monitoring Laboratory, [gml.noaa.gov/ccgg/trends](https://gml.noaa.gov/ccgg/trends/). GISTEMP Team, NASA Goddard Institute for Space Studies, [data.giss.nasa.gov/gistemp](https://data.giss.nasa.gov/gistemp/).

## 1. Settings

The files to fetch. To include an optional extra, delete the `#` at the start of its line.

In [ ]:
from pathlib import Path

OUT_DIR = Path("climate_data")

# name to save as  ->  list of URLs to try, in order
FILES = {
    "co2_mm_gl.csv": [
        "https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_gl.csv",
        "https://gml.noaa.gov/aftp/products/trends/co2/co2_mm_gl.csv",   # backup location
    ],
    "ch4_mm_gl.csv":   ["https://gml.noaa.gov/webdata/ccgg/trends/ch4/ch4_mm_gl.csv"],
    "n2o_mm_gl.csv":   ["https://gml.noaa.gov/webdata/ccgg/trends/n2o/n2o_mm_gl.csv"],
    "GLB.Ts+dSST.csv": ["https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv"],

    # Optional extras:
    # "co2_mm_mlo.csv": ["https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_mm_mlo.csv"],  # Mauna Loa CO2, back to 1958
    # "co2_gr_gl.csv":  ["https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_gr_gl.csv"],   # CO2 annual growth rate
    # "ch4_gr_gl.csv":  ["https://gml.noaa.gov/webdata/ccgg/trends/ch4/ch4_gr_gl.csv"],   # CH4 annual growth rate
    # "n2o_gr_gl.csv":  ["https://gml.noaa.gov/webdata/ccgg/trends/n2o/n2o_gr_gl.csv"],   # N2O annual growth rate
}

## 2. Download

Each file is saved untouched. If a download fails, any copy you already had is left in place.

In [ ]:
import ssl
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

HEADERS = {"User-Agent": "Mozilla/5.0 (personal climate dashboard; python urllib)"}

def download(name, urls, out_dir=OUT_DIR):
    for url in urls:
        try:
            with urlopen(Request(url, headers=HEADERS), timeout=60) as resp:
                data = resp.read()
        except HTTPError as e:
            print(f"  ✗ {name}: HTTP {e.code} from {url}")
            continue
        except URLError as e:
            if "CERTIFICATE_VERIFY_FAILED" in str(e.reason):
                print(f"  ✗ {name}: SSL certificate error. On a Mac, open Finder > Applications > "
                      "Python 3.x and double-click 'Install Certificates.command', then re-run.")
                return False
            print(f"  ✗ {name}: could not connect to {url} ({e.reason})")
            continue
        (out_dir / name).write_bytes(data)
        print(f"  ✓ {name}  ({len(data) / 1024:.0f} KB)")
        return True
    return False

OUT_DIR.mkdir(exist_ok=True)
print(f"Saving to: {OUT_DIR.resolve()}\n")
failed = [name for name, urls in FILES.items() if not download(name, urls)]

if failed:
    print("\nNot downloaded:", ", ".join(failed))
    print("You can open those links in a browser and save the page instead.")
else:
    print("\nAll files downloaded.")

## 3. Quick look

A sanity check: does each file read cleanly, and how current is it? This only reads the files; it never changes them.

In [ ]:
import pandas as pd

MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def read_noaa(path):
    """NOAA GML monthly file: '#' comment header, then year, month, decimal, average, ..."""
    df = pd.read_csv(path, comment="#")
    df.columns = [c.strip().lower() for c in df.columns]
    value_col = "average" if "average" in df.columns else df.columns[3]
    df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=15))
    df["value"] = pd.to_numeric(df[value_col], errors="coerce")
    return df.loc[df["value"] > 0, ["date", "value"]]   # NOAA marks missing months with negative values

def read_gistemp(path):
    """NASA GISTEMP table: one title line, then Year, Jan..Dec, J-D, ...; '***' means not yet available."""
    df = pd.read_csv(path, skiprows=1, na_values=["***", "****"])
    long = df.melt(id_vars="Year", value_vars=MONTHS, var_name="month", value_name="value")
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
    long = long.dropna(subset=["value"])
    long["date"] = pd.to_datetime(long["Year"].astype(int).astype(str) + "-" + long["month"] + "-15",
                                  format="%Y-%b-%d")
    return long.sort_values("date")[["date", "value"]].reset_index(drop=True)

SERIES = {
    "CO₂":             ("co2_mm_gl.csv",   read_noaa,    "ppm"),
    "Methane (CH₄)":   ("ch4_mm_gl.csv",   read_noaa,    "ppb"),
    "N₂O":             ("n2o_mm_gl.csv",   read_noaa,    "ppb"),
    "Temperature":     ("GLB.Ts+dSST.csv", read_gistemp, "°C vs 1951–1980"),
}

data, rows = {}, []
for label, (fname, reader, units) in SERIES.items():
    path = OUT_DIR / fname
    if not path.exists():
        rows.append([label, fname, "missing", "", "", "", units])
        continue
    try:
        df = reader(path)
    except Exception as e:
        rows.append([label, fname, f"could not read: {e}", "", "", "", units])
        continue
    data[label] = df
    rows.append([label, fname, f"{len(df):,} months",
                 df["date"].min().strftime("%b %Y"), df["date"].max().strftime("%b %Y"),
                 round(df["value"].iloc[-1], 2), units])

summary = pd.DataFrame(rows, columns=["Series", "File", "Rows", "From", "To", "Latest value", "Units"])
summary

In [ ]:
import matplotlib.pyplot as plt

INK, INK_2, GRID, SURFACE, LINE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb", "#2a78d6"

fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), facecolor=SURFACE)
for ax, (label, (fname, _, units)) in zip(axes.flat, SERIES.items()):
    ax.set_facecolor(SURFACE)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_2, labelsize=9, length=0)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_title(f"{label}  ·  {units}", loc="left", color=INK, fontsize=11, pad=8)
    if label not in data:
        ax.text(0.5, 0.5, f"{fname} not available", transform=ax.transAxes,
                ha="center", va="center", color=INK_2)
        continue
    df = data[label]
    if label == "Temperature":
        ax.axhline(0, color=INK_2, linewidth=0.8)
    ax.plot(df["date"], df["value"], color=LINE, linewidth=0.8 if label == "Temperature" else 1.2)
    last = df.iloc[-1]
    ax.annotate(f"{last['value']:.2f}  ({last['date']:%b %Y})", (last["date"], last["value"]),
                xytext=(-4, 8), textcoords="offset points", ha="right", color=INK, fontsize=9)

fig.suptitle("Monthly values as downloaded", x=0.01, ha="left", color=INK, fontsize=13)
fig.tight_layout()
plt.show()

## 4. Send the files

Runs in Colab or locally. In Colab it bundles the files into `climate_data.zip` and starts the download in your browser. Locally it tells you where the folder is. Then drag the files (or the zip) into the chat.

In [ ]:
import shutil

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

if colab_files:
    zip_path = shutil.make_archive("climate_data", "zip", OUT_DIR)
    colab_files.download(zip_path)
    print("Downloading climate_data.zip. Unzip it and drag the files into the chat.")
else:
    print("Your files are in:", OUT_DIR.resolve())
    for p in sorted(OUT_DIR.iterdir()):
        print("  ", p.name)